In [ ]:
import os
from urllib.parse import parse_qs

query_string = os.environ.get("QUERY_STRING", "")
params = parse_qs(query_string)

# Parameters (will be overridden by URL query params when running with Voila)
tenantId = params.get("tenantId", [None])[0]
accountId = params.get("entityAccountId", [None])[0]
filter = None
timeRange = params.get("granularity", ["month"])[0]
if timeRange == "day":
    filter = "day"
elif timeRange == "month":
    filter = "month"
elif timeRange == "year":
    filter = "year"
else:
    filter = None  # All time

# Backend base URL comes from environment when running under Voila
# Set CASE_MGMT_BACKEND_URL in the environment; this default is for local dev only.

backendUrl = os.getenv("CASE_MGMT_BACKEND_URL", "http://localhost:3090")
# Service token for authenticated API requests (injected by Voila proxy)
# Voila converts X-Service-Token header to HTTP_X_SERVICE_TOKEN env var
SERVICE_TOKEN = os.getenv('HTTP_X_SERVICE_TOKEN') or os.getenv('SERVICE_TOKEN')
if SERVICE_TOKEN:
    headers = {'Authorization': f'Bearer {SERVICE_TOKEN}'}
else:
    headers = {}
    print("WARNING: No SERVICE_TOKEN found - API calls may fail")

In [ ]:
try:
    import requests
    import pandas as pd
    import plotly.graph_objects as go
    import plotly.express as px
    import plotly.io as pio
    from plotly.subplots import make_subplots
    from IPython.display import display, HTML
    import os
    
    # Set default renderer to basic html/notebook
    pio.renderers.default = "notebook"

    # Output styling
    display(HTML("""
    <style>
        body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; }
        .metric-card { background: white; padding: 15px; border-radius: 8px; box-shadow: 0 1px 3px rgba(0,0,0,0.1); text-align: center; }
        .metric-value { font-size: 24px; font-weight: bold; color: #111827; }
        .metric-label { font-size: 12px; color: #6B7280; text-transform: uppercase; letter-spacing: 0.05em; }
        .grid-container { display: grid; grid-template-columns: repeat(4, 1fr); gap: 15px; margin-bottom: 20px; }
        .table-container { margin-top: 20px; overflow-x: auto; }
        table { width: 100%; border-collapse: collapse; font-size: 14px; }
        th { text-align: left; padding: 12px; border-bottom: 1px solid #e5e7eb; color: #6B7280; font-weight: 600; }
        td { padding: 12px; border-bottom: 1px solid #e5e7eb; color: #111827; }
        tr:last-child td { border-bottom: none; }
    </style>
    """))
except Exception as e:
    print(f"Import Error: {e}")

In [ ]:
# Fetch Data Helper - supports both entity_id and end_to_end_id via backend smart detection
def fetch_transaction_history(id_value, backend_url, time_filter=None):
    """
    Fetch transaction history using the backend endpoint.
    Backend auto-detects if id_value is:
    - UUID format (end_to_end_id): returns single transaction with all 4 entity perspectives
    - Entity ID format (account/counterparty): returns transaction history for that entity
    """
    url = f"{backend_url}/api/v1/jupyter/proxy/transaction-history/{id_value}"
    params = {'tenantId': tenantId}
    if time_filter:
        params['granularity'] = time_filter
    
    try:
        response = requests.get(url, params=params, headers=headers)
        if not response.ok:
            try:
                error_data = response.json()
                error_msg = error_data.get('message', response.text)
                print(f"Request to {url} failed: {response.status_code} {response.reason}")
                print(f'Error message: {error_msg}')
            except Exception:
                print(f"Request to {url} failed: {response.status_code} {response.reason}")
                print('Response body:', response.text)
            return None
        return response.json()
    except Exception as e:
        import traceback
        print(f"Exception during fetch_transaction_history: {e}")
        traceback.print_exc()
        return None
    
def fetch_json(endpoint: str, params: dict):
    url = f"{backendUrl}{endpoint}"
    resp = requests.get(url, params=params, headers=headers)
    resp.raise_for_status()
    return resp.json()

# Main Fetch Logic with Fallback
# Ensure variables exist even when fetch fails
data = None
summary = {}
timeline = []
volume_dist = []
cumulative = []
recent = []
df_timeline = pd.DataFrame()
df_volume = pd.DataFrame()
df_cumulative = pd.DataFrame()
df_recent = pd.DataFrame()

try:
    data = fetch_transaction_history(accountId, backendUrl, filter)
    # 2. If Failed/Empty or Default, try fallback end_to_end_id (Dev environment only)
    if not data:
        # print(f"Trying fallback end_to_end_id: {fallback_end_to_end_id}")
        data = fetch_transaction_history(accountId, backendUrl)
    # print(data)
    if data:
        summary = data.get('summary', {})
        timeline = data.get('timeline', [])
        volume_dist = data.get('volumeDistribution', [])
        # Accept alternative keys
        if not volume_dist:
            volume_dist = data.get('volume_dist') or data.get('volume') or []
        cumulative = data.get('cumulative', [])
        recent = data.get('recentTransactions', [])
        
        df_timeline = pd.DataFrame(timeline)
        if not df_timeline.empty:
            df_timeline['date'] = pd.to_datetime(df_timeline['date'])
            df_timeline = df_timeline.sort_values('date').reset_index(drop=True)

        # Normalise volume distribution from backend (bucketStart/transactionCount)
        df_volume = pd.DataFrame(volume_dist)

        if not df_volume.empty:
            if 'bucketStart' in df_volume.columns:
                df_volume['date'] = pd.to_datetime(df_volume['bucketStart'])
            elif 'date' in df_volume.columns:
                df_volume['date'] = pd.to_datetime(df_volume['date'])
            else:
                df_volume['date'] = pd.NaT

            # Ensure we have a transaction-count column for charts
            if 'transactionCount' in df_volume.columns:
                df_volume['txCount'] = df_volume['transactionCount']
            elif 'bucket_tx_count' in df_volume.columns:
                df_volume['txCount'] = df_volume['bucket_tx_count']
            elif 'txCount' in df_volume.columns:
                df_volume['txCount'] = df_volume['txCount']
            elif 'count' in df_volume.columns:
                df_volume['txCount'] = df_volume['count']
            else:
                df_volume['txCount'] = 0
            
        df_cumulative = pd.DataFrame(cumulative)
        if not df_cumulative.empty:
            df_cumulative['date'] = pd.to_datetime(df_cumulative['date'])
            df_cumulative = df_cumulative.sort_values('date').reset_index(drop=True)
            
        df_recent = pd.DataFrame(recent)
    else:
        display(HTML(f"""
        <div style='color: #ef4444; padding: 20px; border: 2px solid #fee2e2; border-radius: 8px; background: #fef2f2;'>
            <h3 style='margin-top: 0;'>No Data Available</h3>
            <p>Could not fetch transaction data from the backend.</p>
            <p><strong>Backend URL:</strong> {backendUrl}</p>
            <p><strong>Query ID:</strong> {accountId if accountId != 'default_entity_id' else accountId}</p>
            <p>Please check:</p>
            <ul>
                <li>Backend server is running</li>
                <li>The ID exists in the database</li>
                <li>Database connection is working</li>
                <li>Check backend logs for detailed error messages</li>
            </ul>
        </div>
        """))
            
except Exception as e:
    import traceback
    display(HTML(f"<div style='color: red; padding: 20px; border: 1px solid red; border-radius: 5px;'>Error fetching data: {str(e)}</div>"))
    print(f"Backend URL: {backendUrl}")
    traceback.print_exc()


In [ ]:
if data:
    # Display Metrics
    total_vol = f"{summary.get('totalVolume', 0):,.2f}"
    total_tx = f"{summary.get('totalTransactions', 0):,}"
    alerts = f"{summary.get('alertsTriggered', 0)}"
    investigated = f"{summary.get('investigated', 0)}"

    display(HTML(f"""
    <div class=\"grid-container\">
        <div class=\"metric-card\">
            <div class=\"metric-label\">Total Volume</div>
            <div class=\"metric-value\">{total_vol}</div>
        </div>
        <div class=\"metric-card\">
            <div class=\"metric-label\">Total Transactions</div>
            <div class=\"metric-value\">{total_tx}</div>
        </div>
        <div class=\"metric-card\">
            <div class=\"metric-label\">Alerts Triggered</div>
            <div class=\"metric-value\">{alerts}</div>
        </div>
        <div class=\"metric-card\">
            <div class=\"metric-label\">Investigated</div>
            <div class=\"metric-value\">{investigated}</div>
        </div>
    </div>
    """))

In [ ]:

# Ensure timeline always has data
# if df_timeline.empty:
#     from datetime import datetime
#     import pandas as pd
    
#     print("⚠ No timeline data → using fallback")

#     df_timeline = pd.DataFrame({
#         'date': pd.date_range(end=pd.Timestamp.today(), periods=10),
#         'amount': [0]*10,
#         'transactionId': range(10),
#         'isAlerted': [False]*10
#     })

#     df_volume_derived = pd.DataFrame({
#         'date': pd.date_range(end=pd.Timestamp.today(), periods=1),
#         'txCount': [0],
#         'totalAmount': [0.0],
#         'alertCount': [0]
#     })

# required_cols = ['amount', 'transactionId', 'isAlerted']

# for col in required_cols:
#     if col not in df_timeline.columns:
#         df_timeline[col] = 0 if col != 'isAlerted' else False
# if df_cumulative.empty:
#     df_cumulative = pd.DataFrame({
#         'date': df_timeline['date'],
#         'cumulativeAmount': df_timeline['amount'].cumsum(),
#         'cumulativeCount': range(1, len(df_timeline) + 1)
#     })

from datetime import datetime, timedelta
import pandas as pd

# Ensure timeline always has data
if df_timeline.empty:
    print("⚠ No timeline data → using fallback")
    df_timeline = pd.DataFrame({
        'date': pd.date_range(end=pd.Timestamp.today(), periods=10),
        'amount': [0]*10,
        'transactionId': range(10),
        'isAlerted': [False]*10
    })

# Ensure required columns exist
required_cols = ['amount', 'transactionId', 'isAlerted']
for col in required_cols:
    if col not in df_timeline.columns:
        df_timeline[col] = 0 if col != 'isAlerted' else False

# Ensure df_cumulative fallback
if df_cumulative.empty:
    df_cumulative = pd.DataFrame({
        'date': df_timeline['date'],
        'cumulativeAmount': df_timeline['amount'].cumsum(),
        'cumulativeCount': range(1, len(df_timeline) + 1)
    })

# Always derive df_volume_derived from df_timeline (used by Row 3 regardless of data)
_filter = filter if filter in ("day", "month", "year") else "month"
if _filter == "day":
    df_timeline['period'] = df_timeline['date'].dt.normalize()
elif _filter == "year":
    df_timeline['period'] = df_timeline['date'].dt.to_period('Y').dt.to_timestamp()
else:
    df_timeline['period'] = df_timeline['date'].dt.to_period('M').dt.to_timestamp()

df_volume_derived = (
    df_timeline.groupby('period')
    .agg(
        txCount=('transactionId', 'count'),
        totalAmount=('amount', 'sum'),
        alertCount=('isAlerted', 'sum')
    )
    .reset_index()
    .rename(columns={'period': 'date'})
)

# if df_volume.empty:
#     df_volume = pd.DataFrame({
#         'date': df_timeline['date'],
#         'txCount': [0]*len(df_timeline)
#     })

if timeline == []:
    display(HTML("<p style='color: orange;'>No transaction data available — showing empty chart</p>"))
    
if data:
    from datetime import datetime, timedelta
    import calendar
    
    # Sort timeline by date chronologically
    # df_timeline = df_timeline.sort_values('date')
    
    # Determine the full date range from the data
    min_date = df_timeline['date'].min()
    max_date = df_timeline['date'].max()
    
    # Determine frequency based on filter parameter
    # if filter == "day":
    #     freq = 'D'
    #     date_format = '%b %d'
    #     tick_angle = -35
    # elif filter == "year":
    #     freq = 'YS'  # Year start
    #     date_format = '%Y'
    #     tick_angle = 0
    # else:  # month or None (default to month)
    #     freq = 'MS'  # Month start
    #     date_format = '%b %Y'
    #     tick_angle = -45

    # AFTER — freq removed, no longer needed
    if filter == "day":
        date_format = '%b %d'
        tick_angle = -35
    elif filter == "year":
        date_format = '%Y'
        tick_angle = 0
    else:
        date_format = '%b %Y'
        tick_angle = -45
    
    # Generate complete date range based on the filter granularity
    # try:
    #     date_range = pd.date_range(start=min_date.normalize(), end=max_date.normalize(), freq=freq)
    # # For single transaction (or same-period transactions), date_range may be empty
    #     if len(date_range) == 0:
    #     # Snap min_date to the correct period boundary
    #         if filter == "year":
    #             period_start = min_date.to_period('Y').to_timestamp()
    #         elif filter == "day":
    #             period_start = min_date.normalize()
    #         else:  # month
    #             period_start = min_date.to_period('M').to_timestamp()
    #         date_range = pd.DatetimeIndex([period_start])
    # except Exception as e:
    #     print(f"Warning: date_range creation failed: {e}. Using single date.")
    #     date_range = pd.DatetimeIndex([min_date.normalize()])
    
    # df_complete_range = pd.DataFrame({'date': date_range})
    
    # # Prepare transaction data with date normalized to the granularity level
    # df_timeline_agg = df_timeline.copy()
    
    # if filter == "day":
    #     df_timeline_agg['date_key'] = df_timeline_agg['date'].dt.normalize()
    # elif filter == "year":
    #     df_timeline_agg['date_key'] = df_timeline_agg['date'].dt.to_period('Y').dt.to_timestamp()
    # else:  # month
    #     df_timeline_agg['date_key'] = df_timeline_agg['date'].dt.to_period('M').dt.to_timestamp()
    
    # # Aggregate transactions by the selected granularity: sum amounts, count transactions, track alerts
    # df_agg = (
    #     df_timeline_agg.groupby('date_key')
    #     .agg(
    #         amount=('amount', 'sum'),
    #         txCount=('transactionId', 'count'),
    #         hasAlert=('isAlerted', 'max')  # True if any transaction in that period was alerted
    #     )
    #     .reset_index()
    #     .rename(columns={'date_key': 'date'})
    # )
    
    # # Merge complete date range with actual transaction data
    # df_normalized = df_complete_range.merge(df_agg, on='date', how='left')
    
    # # Fill missing values with 0
    # df_normalized['amount'] = df_normalized['amount'].fillna(0)
    # df_normalized['txCount'] = df_normalized['txCount'].fillna(0).astype(int)
    # df_normalized['hasAlert'] = df_normalized['hasAlert'].fillna(False).astype(bool)

    min_date = df_timeline['date'].min()
    max_date = df_timeline['date'].max()

    if filter == "day":
        padding = timedelta(days=0.5)
    elif filter == "year":
        padding = timedelta(days=180)
    else:
        padding = timedelta(days=15)

    x_range_start = (min_date - padding).strftime('%Y-%m-%d')
    x_range_end   = (max_date + padding).strftime('%Y-%m-%d')
    
    # Calculate X-axis range with padding
    # if filter == "day":
    #     padding = timedelta(days=0.5)
    # elif filter == "year":
    #     padding = timedelta(days=180)  # ~6 months
    # else:  # month
    #     padding = timedelta(days=15)
    
    # Derive range from raw df_timeline timestamps, not bucketed date_range
    # min_date = df_timeline['date'].min()
    # max_date = df_timeline['date'].max()
    # x_range_start = (min_date - padding).strftime('%Y-%m-%d')
    # x_range_end   = (max_date + padding).strftime('%Y-%m-%d')
    
    # range_anchor_start = date_range[0]
    # range_anchor_end = date_range[-1]  # last entry — works for both single and multiple
    # x_range_start = (range_anchor_start - padding).strftime('%Y-%m-%d')
    # x_range_end = (range_anchor_end + padding + timedelta(days=30 if filter == 'month' else 1)).strftime('%Y-%m-%d')
    # x_range_start = (min_date.normalize() - padding).strftime('%Y-%m-%d')
    # x_range_end = (max_date.normalize() + padding).strftime('%Y-%m-%d')
    
    # Separate periods with and without alerts for visualization
    # df_alert_periods = df_normalized[df_normalized['hasAlert'] == True]
    # df_normal_periods = df_normalized[(df_normalized['hasAlert'] == False) & (df_normalized['amount'] > 0)]
    
    # Create Charts: 3 rows -> Timeline (dual axis), Cumulative, Volume Distribution
    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=False,
        vertical_spacing=0.14,
        subplot_titles=(
            "Transaction Timeline",
            "Cumulative Value",
            "Volume Distribution"
        ),
        row_heights=[0.4, 0.3, 0.3],
        specs=[
            [{}],  # Row 1: dual y-axis
            [{}],                      # Row 2
            [{}],                      # Row 3
        ],
    )

    # 1. Transaction Timeline - Amount Line (Blue, smooth curve)
    # fig.add_trace(
    #     go.Scatter(
    #         x=df_normalized['date'],
    #         y=df_normalized['amount'],
    #         mode='lines',
    #         name='Amount ($)',
    #         line=dict(color='#3B82F6', width=2.5, shape='spline'),
    #         hovertemplate=(
    #             "<b>%{x|%b %d, %Y}</b><br>" +
    #             "Amount: $%{y:,.2f}<br>" +
    #             "<extra></extra>"
    #         ),
    #         connectgaps=False,
    #     ),
    #     row=1,
    #     col=1,
    #     secondary_y=False,
    # )
    
    # # Add markers for periods with normal transactions (blue dots)
    # if not df_normal_periods.empty:
    #     fig.add_trace(
    #         go.Scatter(
    #             x=df_normal_periods['date'],
    #             y=df_normal_periods['amount'],
    #             mode='markers',
    #             name='Transaction',
    #             marker=dict(size=8, color='#3B82F6', line=dict(width=1.5, color='white')),
    #             hovertemplate=(
    #                 "<b>%{x|%b %d, %Y}</b><br>" +
    #                 "Amount: $%{y:,.2f}<br>" +
    #                 "Status: Normal<br>" +
    #                 "<extra></extra>"
    #             ),
    #             showlegend=True,
    #         ),
    #         row=1,
    #         col=1,
    #         secondary_y=False,
    #     )
    
    # # Add markers for alert periods (red dots)
    # if not df_alert_periods.empty:
    #     fig.add_trace(
    #         go.Scatter(
    #             x=df_alert_periods['date'],
    #             y=df_alert_periods['amount'],
    #             mode='markers',
    #             name='Alert Triggered',
    #             marker=dict(
    #                 size=11, 
    #                 color='#EF4444',
    #                 line=dict(width=2, color='white'),
    #                 symbol='circle'
    #             ),
    #             hovertemplate=(
    #                 "<b>%{x|%b %d, %Y}</b><br>" +
    #                 "Amount: $%{y:,.2f}<br>" +
    #                 "Status: ⚠ ALERT<br>" +
    #                 "<extra></extra>"
    #             ),
    #             showlegend=True,
    #         ),
    #         row=1,
    #         col=1,
    #         secondary_y=False,
    #     )

    # # 1b. Transaction Count Line (Green, smooth curve on right Y-axis)
    # fig.add_trace(
    #     go.Scatter(
    #         x=df_normalized['date'],
    #         y=df_normalized['txCount'],
    #         mode='lines+markers',
    #         name='Transaction Count',
    #         line=dict(color='#10B981', width=2.5, shape='spline'),
    #         marker=dict(size=6, color='#10B981', line=dict(width=1.5, color='white')),
    #         hovertemplate=(
    #             "<b>%{x|%b %d, %Y}</b><br>" +
    #             "Count: %{y}<br>" +
    #             "<extra></extra>"
    #         ),
    #         connectgaps=False,
    #     ),
    #     row=1,
    #     col=1,
    #     secondary_y=True,
    # )

    # 1. Transaction Timeline - per-transaction amount (raw df_timeline)
    # _gap_threshold = timedelta(days=1)

    if _filter == "day":
        _gap_threshold = timedelta(hours=12)   # break if > 12hrs gap within a day view
    elif _filter == "month":
        _gap_threshold = timedelta(days=3)     # break if > 3 days gap within month view
    elif _filter == "year":
        _gap_threshold = timedelta(days=30)    # break if > 30 days gap within year view
    else:
        _gap_threshold = timedelta(days=3)     # default

    _tl_dates = []
    _tl_amounts = []
    _tl_colors = []
    _tl_custom = []

    for i, row in df_timeline.iterrows():
        if _tl_dates:
            prev_date = _tl_dates[-1] if _tl_dates[-1] is not None else df_timeline.iloc[df_timeline.index.get_loc(i)-1]['date']
            if isinstance(prev_date, pd.Timestamp) and (row['date'] - prev_date) > _gap_threshold:
                # Insert null to break the line
                _tl_dates.append(None)
                _tl_amounts.append(None)
                _tl_colors.append('rgba(0,0,0,0)')
                _tl_custom.append('gap')
        _tl_dates.append(row['date'])
        _tl_amounts.append(row['amount'])
        _tl_colors.append('#EF4444' if row['isAlerted'] else '#3B82F6')
        _tl_custom.append('⚠ ALERT' if row['isAlerted'] else 'Normal')

    fig.add_trace(
        go.Scatter(
            x=_tl_dates,
            y=_tl_amounts,
            mode='lines+markers',
            name='Amount ($)',
            line=dict(color='#3B82F6', width=2, shape='linear'),
            fill=None,
            marker=dict(
                size=8,
                color=_tl_colors,
                line=dict(width=1.5, color='white')
            ),
            hovertemplate=(
                "<b>%{x|%b %d, %Y %H:%M}</b><br>" +
                "Amount: $%{y:,.2f}<br>" +
                "Status: %{customdata}<br>" +
                "<extra></extra>"
            ),
            customdata=_tl_custom,
            connectgaps=False,
        ),
        row=1,
        col=1,
        secondary_y=False,
    )
    # fig.add_trace(
    #     go.Scatter(
    #         x=df_timeline['date'],
    #         y=df_timeline['amount'],
    #         mode='lines+markers',
    #         name='Amount ($)',
    #         line=dict(color='#3B82F6', width=2.5, shape='spline'),
    #         marker=dict(
    #             size=8,
    #             color=df_timeline['isAlerted'].map({True: '#EF4444', False: '#3B82F6'}),
    #             line=dict(width=1.5, color='white')
    #         ),
    #         hovertemplate=(
    #             "<b>%{x|%b %d, %Y %H:%M}</b><br>" +
    #             "Amount: $%{y:,.2f}<br>" +
    #             "Status: %{customdata}<br>" +
    #             "<extra></extra>"
    #         ),
    #         customdata=df_timeline['isAlerted'].map({True: '⚠ ALERT', False: 'Normal'}),
    #         connectgaps=False,
    #     ),
    #     row=1,
    #     col=1,
    #     secondary_y=False,
    # )

    # 1b. Transaction Count per day on right Y-axis
    # _day_counts = df_timeline.groupby(
    #     df_timeline['date'].dt.normalize()
    # )['amount'].transform('count')

    # fig.add_trace(
    #     go.Scatter(
    #         x=df_timeline['date'],
    #         y=_day_counts,
    #         mode='lines+markers',
    #         name='Transaction Count',
    #         line=dict(color='#10B981', width=2.5, shape='spline'),
    #         marker=dict(size=6, color='#10B981', line=dict(width=1.5, color='white')),
    #         hovertemplate=(
    #             "<b>%{x|%b %d, %Y}</b><br>" +
    #             "Count: %{y}<br>" +
    #             "<extra></extra>"
    #         ),
    #         connectgaps=False,
    #     ),
    #     row=1,
    #     col=1,
    #     secondary_y=True,
    # )
    # 1b. Transaction Count — grouped by filter granularity
    # 1b. Transaction Count per period — bubble markers with count inside
    # if _filter == "day":
    #     _df_timeline_copy = df_timeline.copy()
    #     _df_timeline_copy['bucket'] = _df_timeline_copy['date'].dt.normalize()
    #     _bucket_label = '%b %d'
    # elif _filter == "year":
    #     _df_timeline_copy = df_timeline.copy()
    #     _df_timeline_copy['bucket'] = _df_timeline_copy['date'].dt.to_period('Y').dt.to_timestamp()
    #     _bucket_label = '%Y'
    # else:  # month
    #     _df_timeline_copy = df_timeline.copy()
    #     _df_timeline_copy['bucket'] = _df_timeline_copy['date'].dt.to_period('M').dt.to_timestamp()
    #     _bucket_label = '%b %Y'

    # _df_day_repr = (
    #     _df_timeline_copy.groupby('bucket')
    #     .agg(
    #         date=('date', 'first'),
    #         dayCount=('transactionId', 'count')
    #     )
    #     .reset_index(drop=True)
    # )

    # _bubble_sizes = _df_day_repr['dayCount'].apply(
    #     lambda c: max(24, min(c * 4, 48))
    # )

    # fig.add_trace(
    #     go.Scatter(
    #         x=_df_day_repr['date'],
    #         y=_df_day_repr['dayCount'],
    #         mode='markers+text',
    #         name='Tx Count (per period)',
    #         marker=dict(
    #             size=_bubble_sizes,
    #             color='rgba(16,185,129,0.15)',
    #             symbol='circle',
    #             line=dict(width=2, color='#10B981')
    #         ),
    #         text=_df_day_repr['dayCount'].astype(str),
    #         textfont=dict(size=10, color='#0D9488'),
    #         textposition='middle center',
    #         hovertemplate=(
    #             "<b>%{x|" + _bucket_label + "}</b><br>" +
    #             "Transactions this period: %{y}<br>" +
    #             "<extra></extra>"
    #         ),
    #     ),
    #     row=1,
    #     col=1,
    #     secondary_y=True,
    # )

    # 2. Cumulative Value (Running Total)
    # df_normalized['cumulativeAmount'] = df_normalized['amount'].cumsum()
    
    # fig.add_trace(
    #     go.Scatter(
    #         x=df_normalized['date'],
    #         y=df_normalized['cumulativeAmount'],
    #         fill='tozeroy',
    #         mode='lines',
    #         name='Cumulative Value',
    #         line=dict(color='#10B981', width=2.5, shape='spline'),
    #         fillcolor='rgba(16, 185, 129, 0.12)',
    #         hovertemplate=(
    #             "<b>%{x|%b %d, %Y}</b><br>" +
    #             "Cumulative: $%{y:,.2f}<br>" +
    #             "<extra></extra>"
    #         ),
    #         connectgaps=False,
    #     ),
    #     row=2,
    #     col=1,
    # )

    # fig.add_trace(
    # go.Scatter(
    #     x=df_cumulative['date'] if not df_cumulative.empty else df_normalized['date'],
    #     y=df_cumulative['cumulativeAmount'] if not df_cumulative.empty else df_normalized['amount'].cumsum(),
    #     fill='tozeroy',
    #     mode='lines+markers',
    #     name='Cumulative Value',
    #     line=dict(color='#10B981', width=2.5, shape='spline'),
    #     fillcolor='rgba(16, 185, 129, 0.12)',
    #     marker=dict(size=5, color='#10B981'),
    #     hovertemplate=(
    #         "<b>%{x|%b %d, %Y %H:%M}</b><br>" +
    #         "Cumulative: $%{y:,.2f}<br>" +
    #         "<extra></extra>"
    #     ),
    #     connectgaps=False,
    # ),
    # row=2, col=1,
    # )

    # fig.add_trace(
    #     go.Scatter(
    #         x=df_cumulative['date'],
    #         y=df_cumulative['cumulativeAmount'],
    #         fill='tozeroy',
    #         mode='lines+markers',
    #         name='Cumulative Value',
    #         line=dict(color='#10B981', width=2.5, shape='spline'),
    #         fillcolor='rgba(16, 185, 129, 0.12)',
    #         marker=dict(size=5, color='#10B981'),
    #         hovertemplate=(
    #             "<b>%{x|%b %d, %Y %H:%M}</b><br>" +
    #             "Cumulative: $%{y:,.2f}<br>" +
    #             "<extra></extra>"
    #         ),
    #         connectgaps=False,
    #     ),
    #     row=2, col=1,
    # )

    fig.add_trace(
        go.Scatter(
            x=df_cumulative['date'],
            y=df_cumulative['cumulativeAmount'],
            fill='tozeroy',
            mode='lines+markers',
            name='Cumulative Value',
            line=dict(color='#0D9488', width=2.5, shape='spline'),
            fillcolor='rgba(13, 148, 136, 0.18)',
            marker=dict(
                size=7,
                color='#0D9488',
                line=dict(width=1.5, color='white')
            ),
            hovertemplate=(
                "<b>%{x|%b %d, %Y %H:%M}</b><br>" +
                "Cumulative: $%{y:,.2f}<br>" +
                "Txn #%{customdata}<br>" +
                "<extra></extra>"
            ),
            customdata=df_cumulative['cumulativeCount'] if 'cumulativeCount' in df_cumulative.columns else list(range(1, len(df_cumulative)+1)),
            connectgaps=False,
        ),
        row=2, col=1,
    )

    # 3. Transaction Volume Distribution (Bar chart of counts)
    # fig.add_trace(
    #     go.Bar(
    #         x=df_normalized['date'],
    #         y=df_normalized['txCount'],
    #         name='Volume Distribution',
    #         marker=dict(
    #             color='#6366F1',
    #             opacity=0.85,
    #             line=dict(width=0)
    #         ),
    #         hovertemplate=(
    #             "<b>%{x|%b %d, %Y}</b><br>" +
    #             "Transactions: %{y}<br>" +
    #             "<extra></extra>"
    #         ),
    #     ),
    #     row=3,
    #     col=1,
    # )

    # 3. Volume Distribution — derived from df_timeline
    fig.add_trace(
        go.Bar(
            x=df_volume_derived['date'],
            y=df_volume_derived['txCount'],
            name='Volume Distribution',
            marker=dict(
                color='#6366F1',
                opacity=0.85,
                line=dict(width=0)
            ),
            customdata=df_volume_derived[['totalAmount', 'alertCount']].values,
            hovertemplate=(
                "<b>%{x|%b %d, %Y}</b><br>" +
                "Transactions: %{y}<br>" +
                "Total amount: $%{customdata[0]:,.2f}<br>" +
                "Alerts: %{customdata[1]}<br>" +
                "<extra></extra>"
            ),
        ),
        row=3,
        col=1,
    )

    # Style the subplot titles (annotations) — bold, larger, consistent colour
    for ann in fig.layout.annotations:
        ann.update(
            font=dict(size=13, color='#374151', family="-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif"),
            xanchor='center',
        )

    # Layout & axes styling - Modern fintech dashboard style
    fig.update_layout(
        autosize=True,
        height=1040,
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="top",
            y=1.12,          # pushed well below the bottom chart's x-axis labels - Adjusted from 1.08 to 1.12
            xanchor="center",
            x=0.5,
            bgcolor="rgba(255, 255, 255, 0.9)",
            bordercolor="#E5E7EB",
            borderwidth=1,
            font=dict(size=11),
            traceorder="normal",
        ),
        # Extra bottom margin to give the legend clear space beneath the last x-axis
        margin=dict(l=70, r=100, t=90, b=130),
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        hovermode='x unified',
        font=dict(
            family="-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif",
            color="#111827"
        ),
    )

    # Grid lines with subtle styling
    fig.update_xaxes(
        showgrid=True, 
        gridwidth=1, 
        gridcolor='rgba(229, 231, 235, 0.6)',
        showline=True,
        linewidth=1,
        linecolor='#D1D5DB',
        mirror=False
    )
    fig.update_yaxes(
        showgrid=True, 
        gridwidth=1, 
        gridcolor='rgba(229, 231, 235, 0.6)',
        showline=True,
        linewidth=1,
        linecolor='#D1D5DB',
        mirror=False,
        zeroline=True,
        zerolinewidth=1,
        zerolinecolor='#D1D5DB'
    )

    # Shared x-axis config helper
    # shared_xaxis_cfg = dict(
    #     tickformat=date_format,
    #     tickangle=tick_angle,
    #     tickmode='auto',
    #     tickfont=dict(size=9, color='#6B7280'),
    #     title_font=dict(size=11, color='#6B7280'),
    #     fixedrange=False,
    #     showspikes=True,
    #     spikemode='across',
    #     spikesnap='cursor',
    #     spikecolor='#9CA3AF',
    #     spikethickness=1,
    #     nticks=6
    # )

    # # Row 1: Transaction Timeline
    # fig.update_xaxes(title_text="<b>Date</b>", row=1, col=1,
    #                  range=[x_range_start, x_range_end], **shared_xaxis_cfg)
    # fig.update_yaxes(title_text="<b>Amount ($)</b>", row=1, col=1, secondary_y=False,
    #                  title_font=dict(size=11, color='#3B82F6'))
    # fig.update_yaxes(title_text="<b>Tx Count</b>", row=1, col=1, secondary_y=True,
    #                  title_font=dict(size=11, color='#10B981'))

    # # Row 2: Cumulative
    # fig.update_xaxes(title_text="<b>Date</b>", row=2, col=1,
    #                  range=[x_range_start, x_range_end], **shared_xaxis_cfg)
    # fig.update_yaxes(title_text="<b>Cumulative ($)</b>", row=2, col=1,
    #                  title_font=dict(size=11, color='#10B981'))

    # # Row 3: Volume Distribution
    # fig.update_xaxes(title_text="<b>Date</b>", row=3, col=1,
    #                  range=[x_range_start, x_range_end], **shared_xaxis_cfg)
    # fig.update_yaxes(title_text="<b>Count</b>", row=3, col=1,
    #                  title_font=dict(size=11, color='#6366F1'))

    # Shared x-axis config — used for Row 3 (bucketed periods only)
    shared_xaxis_cfg = dict(
        tickformat=date_format,
        tickangle=tick_angle,
        tickmode='auto',
        tickfont=dict(size=9, color='#6B7280'),
        title_font=dict(size=11, color='#6B7280'),
        fixedrange=False,
        showspikes=True,
        spikemode='across',
        spikesnap='cursor',
        spikecolor='#9CA3AF',
        spikethickness=1,
        nticks=6
    )

    # Timestamp x-axis config — used for Row 1 & 2 (exact per-transaction timestamps)
    if filter == "day":
        ts_tickformat = '%b %d %H:%M'   # transactions within a day — need time
        ts_tickangle = -35
        ts_nticks = 10
    elif filter == "year":
        ts_tickformat = '%b %d, %Y'     # transactions across years — need full date
        ts_tickangle = -35
        ts_nticks = 8
    else:  # month (default)
        ts_tickformat = '%b %d'         # transactions within month range — date sufficient
        ts_tickangle = -35
        ts_nticks = 8

    timestamp_xaxis_cfg = dict(
        tickformat=ts_tickformat,
        tickangle=ts_tickangle,
        tickmode='auto',
        tickfont=dict(size=9, color='#6B7280'),
        title_font=dict(size=11, color='#6B7280'),
        fixedrange=False,
        showspikes=True,
        spikemode='across',
        spikesnap='cursor',
        spikecolor='#9CA3AF',
        spikethickness=1,
        nticks=ts_nticks
    )

    # Row 1: Transaction Timeline — exact timestamps
    # fig.update_xaxes(title_text="<b>Date</b>", row=1, col=1,
    #                  range=[x_range_start, x_range_end], **timestamp_xaxis_cfg)
    # fig.update_yaxes(title_text="<b>Amount ($)</b>", row=1, col=1, secondary_y=False,
    #                  title_font=dict(size=11, color='#3B82F6'))
    # fig.update_yaxes(title_text="<b>Tx Count</b>", row=1, col=1, secondary_y=True,
    #                  title_font=dict(size=11, color='#10B981'))
    _tl_padding = timedelta(days=1)
    _tl_x_start = (df_timeline['date'].min() - _tl_padding).strftime('%Y-%m-%d')
    _tl_x_end   = (df_timeline['date'].max() + _tl_padding).strftime('%Y-%m-%d')

    fig.update_xaxes(
        title_text="<b>Date</b>",
        row=1, col=1,
        range=[_tl_x_start, _tl_x_end],
        **timestamp_xaxis_cfg
    )
    fig.update_yaxes(
        title_text="<b>Amount ($)</b>",
        row=1, col=1,
        secondary_y=False,
        tickformat='$,.0f',
        title_font=dict(size=11, color='#3B82F6'),
        tickfont=dict(color='#3B82F6')
    )
    # fig.update_yaxes(
    #     title_text="<b>Tx Count</b>",
    #     row=1, col=1,
    #     secondary_y=True,
    #     tickformat='d',
    #     dtick=1,
    #     title_font=dict(size=11, color='#10B981'),
    #     tickfont=dict(color='#10B981')
    # )

    # Row 2: Cumulative — exact timestamps
    # fig.update_xaxes(title_text="<b>Date</b>", row=2, col=1,
    #                  range=[x_range_start, x_range_end], **timestamp_xaxis_cfg)
    # fig.update_yaxes(title_text="<b>Cumulative ($)</b>", row=2, col=1,
    #                  title_font=dict(size=11, color='#10B981'))

    # Row 2: Cumulative — exact timestamps, tighter range
    _cum_min = df_cumulative['date'].min()
    _cum_max = df_cumulative['date'].max()
    _cum_padding = timedelta(days=1)
    _cum_x_start = (_cum_min - _cum_padding).strftime('%Y-%m-%d')
    _cum_x_end   = (_cum_max + _cum_padding).strftime('%Y-%m-%d')

    fig.update_xaxes(
        title_text="<b>Date</b>",
        row=2, col=1,
        range=[_cum_x_start, _cum_x_end],
        **timestamp_xaxis_cfg
    )
    fig.update_yaxes(
        title_text="<b>Cumulative ($)</b>",
        row=2, col=1,
        tickformat='$,.0f',
        title_font=dict(size=11, color='#0D9488'),
        tickfont=dict(color='#0D9488')
    )

    # Row 3: Volume Distribution — bucketed, uses shared_xaxis_cfg
    fig.update_xaxes(title_text="<b>Date</b>", row=3, col=1,
                     range=[x_range_start, x_range_end], **shared_xaxis_cfg)
    fig.update_yaxes(title_text="<b>Count</b>", row=3, col=1,
                     title_font=dict(size=11, color='#6366F1'))

    # Spike lines on y-axes too
    fig.update_yaxes(showspikes=True, spikemode='across', spikesnap='cursor',
                     spikecolor='#9CA3AF', spikethickness=1)

    fig.show()

In [ ]:
# Fetch Benford's Law Data and Create Analysis Chart
try:
    from datetime import datetime, timedelta
    end_date = datetime.now()
    start_date = end_date - timedelta(days=90)

    benford_url = f"{backendUrl}/api/v1/jupyter/proxy/lake/analytics/benford/account/{accountId}"
    benford_params = {
        'tenantId': tenantId,
        'from': start_date.strftime('%Y-%m-%d'),
        'to': end_date.strftime('%Y-%m-%d')
    }

    try:
        benford_response = requests.get(benford_url, params=benford_params, headers=headers, timeout=20)
        
        if not benford_response.ok:
            print(f"Benford request failed: {benford_response.status_code} {benford_response.reason}")
            try:
                print('Response body:', benford_response.text)
            except Exception:
                pass
            benford_data = None
        else:
            benford_response.raise_for_status()
            benford_data = benford_response.json()

    except Exception as e:
        import traceback
        print("Exception during Benford fetch:", e)
        traceback.print_exc()
        benford_data = None

    # Process benford_data if present
    expected_benford = []
    actual_benford = []
    digits = [1,2,3,4,5,6,7,8,9]

    if benford_data and isinstance(benford_data, dict) and 'expected' in benford_data and 'actual' in benford_data:
        exp_dict = benford_data['expected']
        act_dict = benford_data['actual']
        sample_size = float(benford_data.get('sampleSize', 0) or 0)
        for d in digits:
            d_str = str(d)
            expected_benford.append(float(exp_dict.get(d_str, 0)) * 100)
            count = float(act_dict.get(d_str, 0))
            actual_benford.append((count / sample_size * 100) if sample_size > 0 else 0)
    else:
        if benford_data is None:
            print('No Benford data returned from backend')
        else:
            print('Benford response format unexpected:', type(benford_data))

    # Ensure arrays of length 9 for plotting
    if len(expected_benford) != 9:
        expected_benford = [30.1, 17.6, 12.5, 9.7, 7.9, 6.7, 5.8, 5.1, 4.6]
    if len(actual_benford) != 9:
        actual_benford = [0] * 9

except Exception as e:
    import traceback
    print(f"Benford analysis error: {e}")
    traceback.print_exc()


In [ ]:
# Benford chart and table (always render; fallbacks applied earlier)
try:
    import plotly.graph_objects as go
    import pandas as pd

    digits = [1, 2, 3, 4, 5, 6, 7, 8, 9]
    try:
        exp = expected_benford
        act = actual_benford
    except NameError:
        exp = [30.1, 17.6, 12.5, 9.7, 7.9, 6.7, 5.8, 5.1, 4.6]
        act = [0] * 9

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=digits,
        y=exp,
        name="Expected (Benford's Law)",
        marker=dict(color='#10B981', opacity=0.85, line=dict(width=0)),
        hovertemplate="Digit %{x}<br>Expected: %{y:.1f}%<extra></extra>",
    ))
    fig.add_trace(go.Bar(
        x=digits,
        y=act,
        name='Actual Distribution',
        marker=dict(color='#6366F1', opacity=0.85, line=dict(width=0)),
        hovertemplate="Digit %{x}<br>Actual: %{y:.1f}%<extra></extra>",
    ))

    fig.update_layout(
        autosize=True,
        barmode='group',
        bargap=0.18,
        bargroupgap=0.06,
        title=dict(
            text="Benford's Law Distribution Analysis",
            font=dict(size=13, color='#374151',
                      family="-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif"),
            x=0.5,
            xanchor='center',
        ),
        xaxis=dict(
            title=dict(text='<b>First Digit</b>', font=dict(size=11, color='#6B7280')),
            tickmode='array',
            tickvals=digits,
            tickfont=dict(size=11, color='#374151'),
            showgrid=False,
            showline=True,
            linewidth=1,
            linecolor='#D1D5DB',
        ),
        yaxis=dict(
            title=dict(text='<b>Frequency (%)</b>', font=dict(size=11, color='#6B7280')),
            showgrid=True,
            gridwidth=1,
            gridcolor='rgba(229, 231, 235, 0.6)',
            zeroline=True,
            zerolinewidth=1,
            zerolinecolor='#D1D5DB',
            showline=True,
            linewidth=1,
            linecolor='#D1D5DB',
            tickfont=dict(size=10, color='#6B7280'),
        ),
        height=440,
        margin=dict(l=60, r=60, t=70, b=80),
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        hovermode='x',
        font=dict(
            family="-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif",
            color="#111827",
        ),
    )
    fig.update_layout(
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.30,
            xanchor="center",
            x=0.5,
            bgcolor="rgba(255, 255, 255, 0.9)",
            bordercolor="#E5E7EB",
            borderwidth=1,
            font=dict(size=11),
        )
    )

    fig.show()

    # Summary table
    try:
        df_b = pd.DataFrame({
            'Digit': digits,
            'Expected (%)': [round(v, 1) for v in exp],
            'Actual (%)': [round(v, 1) for v in act],
            'Δ (pp)': [round(a - e, 1) for a, e in zip(act, exp)],
        })
        display(df_b.style.hide_index().format({'Expected (%)': '{:.1f}', 'Actual (%)': '{:.1f}', 'Δ (pp)': '{:+.1f}'}))
    except Exception:
        pass

except Exception as e:
    import traceback
    print('Error rendering Benford chart/table:', e)
    traceback.print_exc()


In [ ]:
if df_recent.empty:
    print("⚠ No recent transactions → showing fallback")

    df_recent = pd.DataFrame({
        'date': pd.date_range(end=pd.Timestamp.today(), periods=5),
        'type': ['—']*5,
        'counterparty': ['—']*5,
        'amount': [0]*5,
        'currency': ['—']*5
    })
# if data and not df_recent.empty:
if data:
    display(HTML("<h3>Recent Transactions</h3>"))
    # Select and rename columns for display
    cols = ['date', 'type', 'counterparty', 'amount', 'currency']
    # Ensure cols exist
    valid_cols = [c for c in cols if c in df_recent.columns]
    
    display_df = df_recent[valid_cols].copy()

    if 'date' in display_df.columns:
        display_df['date'] = pd.to_datetime(display_df['date']).dt.strftime('%Y-%m-%d %H:%M:%S')

    # if 'amount' in display_df.columns:
    #     display_df['amount'] = display_df['amount'].apply(
    #     lambda x: f"{x:,.2f}" if pd.notnull(x) else "—"
    # )
    # Format Amount
    if 'amount' in display_df.columns:
        display_df['amount'] = display_df['amount'].apply(lambda x: f"{x:,.2f}")

    rename_map = {
        'date': 'DateTime',
        'type': 'Type',
        'counterparty': 'CounterParty',
        'amount': 'Amount',
        'currency': 'Currency',
    }

    display_df = display_df[[c for c in rename_map.keys() if c in display_df.columns]].rename(columns=rename_map)


    # Render HTML table with styling
    html = display_df.to_html(index=False, classes='table')
    display(HTML(f"<div class='table-container'>{html}</div>"))